<a href="https://colab.research.google.com/github/varunpatelspace/Breast-Cancer-Classification/blob/main/Breast_Cancer_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Problem define ?
- Input (X): Cell measurements ke numerical features.
- Target (y): Benign ya malignant class.
- Task: Binary classification.
- Primary evaluation: Recall, precision, F1,      ROC-AUC; medical context mein false negatives ko specially investigate karenge.
- Baseline: Simple Logistic Regression.
Model ka goal sirf high accuracy achieve karna nahi hai. Humein yeh bhi dekhna hai ki kaunsi class miss ho rahi hai aur unseen data par performance kaisi hai.

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

dataset =

X = dataset.data
y = dataset.target

print("Dataset shape:", X.shape)
print("Feature names:", X.columns.tolist())
print("Target counts:")
print(y.value_counts())

print(X.head())

In [ ]:
# 1 = malignant, 0 = benign
y = (dataset.target == 0).astype(int)

print("Missing values:", X.isna().sum().sum())
print("Duplicate rows:", X.duplicated().sum())
print("Malignant cases:", y.sum())
print("Benign cases:", (y == 0).sum())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=3000))
])

baseline.fit(X_train, y_train)

print("Baseline trained successfully")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scores = cross_validate(
    baseline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "roc_auc": "roc_auc",
        "recall": "recall",
        "precision": "precision",
        "f1": "f1"
    }
)

for metric in ["roc_auc", "recall", "precision", "f1"]:
    values = scores[f"test_{metric}"]
    print(
        f"{metric}: {values.mean():.3f} "
        f"+/- {values.std():.3f}"
    )

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100]
}

search = GridSearchCV(
    estimator=baseline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    refit=True
)

search.fit(X_train, y_train)

print("Best parameters:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 4))

best_model = search.best_estimator_

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Benign", "Malignant"],
    zero_division=0
))

print("Test ROC-AUC:", roc_auc_score(y_test, y_prob))
print("Test Average Precision:", average_precision_score(y_test, y_prob))

In [ ]:
import joblib

joblib.dump(best_model, "breast_cancer_model.joblib")

print("Model saved successfully")

In [ ]:
loaded_model = joblib.load("breast_cancer_model.joblib")

sample = X_test.iloc[[0]]
probability = loaded_model.predict_proba(sample)[0, 1]

print("Predicted malignant probability:", probability)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Load data
data = load_breast_cancer(as_frame=True)
X = data.data

# 1 = Malignant, 0 = Benign
y = (data.target == 0).astype(int)

# Same split for all models
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Features:", X.shape[1])
print("Target mapping: 1 = Malignant, 0 = Benign")


In [ ]:
models = {
    "Logistic Regression": make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=5000, random_state=42)
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),

    "SVM (RBF)": make_pipeline(
        StandardScaler(),
        SVC(
            C=1.0,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        )
    ),

    "KNN": make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=5)
    ),

    "Hist Gradient Boosting": HistGradientBoostingClassifier(
        max_iter=150,
        random_state=42
    )
}


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv_results = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Recall": scores["test_recall"].mean(),
        "CV F1": scores["test_f1"].mean(),
        "CV ROC-AUC": scores["test_roc_auc"].mean()
    })

comparison = pd.DataFrame(cv_results)

# Prioritize malignant-case recall, then F1
comparison = comparison.sort_values(
    ["CV Recall", "CV F1", "CV ROC-AUC"],
    ascending=False
).reset_index(drop=True)

display(comparison.round(4))


In [ ]:
test_results = []
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model

    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]

    test_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Error Rate": 1 - accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1 Score": f1_score(y_test, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, prob)
    })

test_comparison = pd.DataFrame(test_results)
best_name = comparison.iloc[0]["Model"]
best_model = trained_models[best_name]

display(
    test_comparison.sort_values(
        "Error Rate"
    ).reset_index(drop=True).round(4)
)

print("Selected model based on training CV:", best_name)
print("\nClassification Report:")
print(classification_report(
    y_test,
    best_model.predict(X_test),
    target_names=["Benign", "Malignant"],
    zero_division=0
))

cm = confusion_matrix(y_test, best_model.predict(X_test))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign", "Malignant"],
    yticklabels=["Benign", "Malignant"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix - {best_name}")
plt.show()


In [ ]:

# Compare test-set error and recall

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

error_data = test_comparison.sort_values("Error Rate")

sns.barplot(
    data=error_data,
    x="Error Rate",
    y="Model",
    ax=axes[0]
)
axes[0].set_title("Test Error Rate (Lower is Better)")
axes[0].set_xlabel("Error Rate")

recall_data = test_comparison.sort_values("Recall", ascending=False)

sns.barplot(
    data=recall_data,
    x="Recall",
    y="Model",
    ax=axes[1]
)
axes[1].set_title("Malignant Recall (Higher is Better)")
axes[1].set_xlabel("Recall")

plt.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score
)

sns.set_theme(style="whitegrid")

def plot_metric(metric, title, ylabel, ascending=False):
    df = test_comparison.sort_values(
        metric, ascending=ascending
    )

    plt.figure(figsize=(10, 5))
    ax = sns.barplot(data=df, x="Model", y=metric)
    ax.set_title(title, fontsize=15)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylim(0, 1.05)

    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", padding=3, fontsize=8)

    plt.tight_layout()
    plt.show()

plot_metric("Accuracy", "Model Accuracy Comparison",
            "Accuracy Score")


In [ ]:

plot_metric("Error Rate", "Model Error Rate Comparison",
            "Error Rate", ascending=True)


In [ ]:

metrics = ["Precision", "Recall", "F1 Score"]

df_plot = test_comparison.melt(
    id_vars="Model",
    value_vars=metrics,
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=df_plot,
    x="Model",
    y="Score",
    hue="Metric"
)
plt.title("Precision vs Recall vs F1-Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(9, 7))

for name, model in trained_models.items():
    probabilities = model.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, probabilities)
    roc_auc = auc(fpr, tpr)

    plt.plot(
        fpr, tpr,
        label=f"{name} (AUC = {roc_auc:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Malignant Recall)")
plt.title("ROC Curve - All Models")
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()


In [ ]:

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(trained_models.items()):
    pred = model.predict(X_test)
    cm = confusion_matrix(y_test, pred, labels=[0, 1])

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=axes[i],
        xticklabels=["Benign", "Malignant"],
        yticklabels=["Benign", "Malignant"]
    )

    axes[i].set_title(name)
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Actual")

for j in range(len(trained_models), len(axes)):
    axes[j].axis("off")

plt.suptitle("Confusion Matrix Comparison", fontsize=17)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(9, 7))

for name, model in trained_models.items():
    probabilities = model.predict_proba(X_test)[:, 1]

    precision, recall, _ = precision_recall_curve(
        y_test, probabilities
    )
    ap = average_precision_score(y_test, probabilities)

    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

cv_plot = comparison.melt(
    id_vars="Model",
    value_vars=["CV Accuracy", "CV Recall", "CV F1", "CV ROC-AUC"],
    var_name="Metric",
    value_name="Score"
)

plt.figure(figsize=(12, 6))
sns.barplot(
    data=cv_plot,
    x="Model",
    y="Score",
    hue="Metric"
)
plt.title("5-Fold Cross-Validation Model Comparison")
plt.ylim(0, 1.05)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:

# STEP 5: Interactive user input form

import ipywidgets as widgets
from IPython.display import display, clear_output

# Default values are feature medians from the dataset
feature_inputs = {}

for feature in X.columns:
    feature_inputs[feature] = widgets.FloatText(
        value=float(X[feature].median()),
        description=feature,
        style={"description_width": "initial"},
        layout=widgets.Layout(width="650px")
    )

input_button = widgets.Button(
    description="Predict Tumor Class",
    button_style="primary",
    icon="check"
)

output = widgets.Output()

# Select which trained model to use
model_selector = widgets.Dropdown(
    options=list(trained_models.keys()),
    value=best_name,
    description="Model:",
    layout=widgets.Layout(width="400px")
)

def predict_tumor(_):
    with output:
        clear_output(wait=True)

        values = {
            feature: [widget.value]
            for feature, widget in feature_inputs.items()
        }

        input_df = pd.DataFrame(values, columns=X.columns)

        selected_model = trained_models[model_selector.value]
        prediction = int(selected_model.predict(input_df)[0])
        probabilities = selected_model.predict_proba(input_df)[0]

        malignant_index = list(selected_model.classes_).index(1)
        malignant_probability = probabilities[malignant_index]

        result = "MALIGNANT" if prediction == 1 else "BENIGN"

        print("Selected Model:", model_selector.value)
        print("Predicted Class:", result)
        print(f"Estimated Malignant Probability: {malignant_probability:.2%}")

        print("\nNote: This is a machine-learning demonstration,")
        print("not a medical diagnosis.")

input_button.on_click(predict_tumor)

display(model_selector)

# Display 30 feature input fields
display(widgets.VBox(list(feature_inputs.values())))

display(input_button, output)
